# LatentSync 1.6
Official: https://github.com/bytedance/LatentSync . Upstream recommends at least 18GB VRAM. Setup uses official requirements and checkpoints; Colab package conflicts may require a fresh runtime.
Run cells in order on a fresh GPU runtime. Use a single visible face and an isolated vocal stem. Input offsets are independent; final output contains test audio, not the full music mix. Not GPU-tested.

In [ ]:
import os, subprocess, sys, shutil, time
from pathlib import Path
from google.colab import drive, files
from IPython.display import Video, display
def run(*args, cwd=None):
    subprocess.run([str(x) for x in args], cwd=cwd, check=True)
run('nvidia-smi')
drive.mount('/content/drive')
root=Path('/content/drive/MyDrive/Alina_LipSync')
root.mkdir(parents=True, exist_ok=True)
run('apt-get','-qq','update')
run('apt-get','-qq','install','-y','ffmpeg')


In [ ]:
repo=Path('/content/LatentSync')
if not repo.exists(): run('git','clone','https://github.com/bytedance/LatentSync.git',repo)
run(sys.executable,'-m','pip','install','-r',repo/'requirements.txt')
run(sys.executable,'-m','pip','install','huggingface_hub')
from huggingface_hub import hf_hub_download
weights=root/'LatentSync_weights'; weights.mkdir(exist_ok=True)
for f in ['latentsync_unet.pt','whisper/tiny.pt']:
    hf_hub_download('ByteDance/LatentSync-1.6',f,local_dir=weights)
if not (repo/'checkpoints').exists(): (repo/'checkpoints').symlink_to(weights)
else:
    for f in ['latentsync_unet.pt','whisper/tiny.pt']:
        dst=repo/'checkpoints'/f; dst.parent.mkdir(parents=True,exist_ok=True)
        if not dst.exists(): dst.symlink_to(weights/f)
run(sys.executable,'-m','scripts.inference','--help',cwd=repo)

In [ ]:
VIDEO_PATH = '' #@param {type:"string"}
AUDIO_PATH = '' #@param {type:"string"}
VIDEO_START = 0.0 #@param {type:"number"}
AUDIO_START = 0.0 #@param {type:"number"}
DURATION = 8.0 #@param {type:"number"}
def choose(path):
    if path:
        p=Path(path).resolve()
    else:
        uploaded=files.upload()
        assert len(uploaded)==1, 'Upload exactly one file'
        p=Path(next(iter(uploaded))).resolve()
    assert p.is_file(), p
    return p
assert DURATION>0 and VIDEO_START>=0 and AUDIO_START>=0
video=choose(VIDEO_PATH)
audio=choose(AUDIO_PATH)
work=Path('/content/alina_inputs'); work.mkdir(exist_ok=True)
v=work/'video.mp4'; a=work/'vocals.wav'
run('ffmpeg','-y','-ss',VIDEO_START,'-i',video,'-t',DURATION,'-an','-vf','fps=25','-c:v','libx264','-pix_fmt','yuv420p',v)
run('ffmpeg','-y','-ss',AUDIO_START,'-i',audio,'-t',DURATION,'-vn','-ac','1','-ar','16000',a)


In [ ]:
STEPS=20 #@param {type:"integer"}
GUIDANCE=1.5 #@param {type:"number"}
out=root/'latentsync_output.mp4'
t=time.perf_counter()
run(sys.executable,'-m','scripts.inference','--unet_config_path','configs/unet/stage2_512.yaml','--inference_ckpt_path','checkpoints/latentsync_unet.pt','--inference_steps',STEPS,'--guidance_scale',GUIDANCE,'--enable_deepcache','--video_path',v,'--audio_path',a,'--video_out_path',out,cwd=repo)
print('Seconds:',time.perf_counter()-t)
assert out.is_file(), out
print('Saved:',out)
display(Video(str(out),embed=True))
# Uncomment to download:
# files.download(str(out))